# Entendimiento del Problema — Admisiones de Posgrado

**Autor:** Equipo SCO

**Fecha:** 2026-08-19

**Descripción:**
Notebook del Issue #1 "Descarga de los datos". Cubre el entendimiento del problema,
la lectura reproducible del dataset RAW y una exploración inicial mínima. No incluye
limpieza, feature engineering ni modelamiento (etapas posteriores del curso).


## 🎯 Entendimiento del problema (respuestas requeridas por el Issue #1)

### 1. ¿Cuál es el objetivo del problema?
Estimar la probabilidad de admisión de un candidato a un programa de posgrado
(variable `Chance of Admit`, en el rango 0–1) a partir de su perfil académico, para que
los estudiantes tengan una idea justa de sus posibilidades de ser admitidos.
*(Respaldado por `data/01_raw/Informacion.txt`.)*

### 2. ¿Cómo se usará su solución?
**No documentado** en el repositorio. **Supuesto:** herramienta de consulta *offline* para
que un estudiante estime su chance de admisión antes de postular; no hay indicio de
integración en un sistema en producción.

### 3. ¿Cuáles son las soluciones actuales (si las hay)?
**No documentadas** en el repositorio: ni `README.md`, ni `AGENTS.md` ni
`Informacion.txt` mencionan una solución existente para este problema.

### 4. ¿Cómo se debe enmarcar este problema?
- Tipo: **regresión supervisada**. `Informacion.txt` lo deja como
  "REGRESION O CLASIFICACION"; la inspección del dato (target continuo en [0, 1]) resuelve a regresión.
- Modo: **offline / batch**. **No documentado**; **supuesto** (no hay indicio de
  streaming ni de inferencia en tiempo real).

### 5. ¿Cómo se debe medir el desempeño? (primera intuición)
Al ser regresión con target en escala 0–1, se propone el **RMSE** (raíz del error cuadrático
medio) como métrica principal y el **MAE** como complementaria; ambas miden el error de
predicción en la misma escala que el target. R² como referencia adicional.

### 6. ¿La medida de desempeño está alineada con el objetivo?
Sí: el objetivo es estimar una probabilidad y RMSE/MAE cuantifican directamente la distancia
entre la probabilidad predicha y la real, en la escala 0–1 del target.

### 7. ¿Cuál sería el desempeño mínimo necesario?
Superar un baseline trivial de "predecir siempre la media del target" (media ≈ 0.733). Ese
baseline tendría un RMSE ≈ desviación estándar del target ≈ 0.142; un modelo aporta valor
solo si logra un RMSE menor. *(Valores calculados sobre el RAW en este notebook.)*

### 8. ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencia o herramientas?
El dataset coincide con el conocido *Graduate Admissions*, ampliamente usado en ejercicios de
regresión; es posible reutilizar enfoques y herramientas públicas aplicadas a ese dataset
(regresión lineal, modelos basados en árboles, etc.). **Atribución/origen no confirmado por
el repositorio** (ver Referencias).

### 9. ¿Hay experiencia del problema disponible?
**No documentada** en el repositorio.

### 10. ¿Cómo se puede resolver el problema manualmente?
Sin modelo: comparar el perfil del candidato (CGPA, GRE, TOEFL) contra estadísticos o
umbrales históricos (por ejemplo, el perfil promedio de los admitidos) y ordenar a los
candidatos por esas variables para asignar una probabilidad aproximada según su posición relativa.

### 11. Listado de supuestos (hasta este momento)
- El dataset representa datos históricos de admisiones de posgrado.
- La variable objetivo `Chance of Admit` es una probabilidad continua en [0, 1].
- Las celdas vacías del CSV son valores faltantes (no ceros) y se tratarán en etapas posteriores.
- Los rangos de cada variable son los documentados en `Informacion.txt`.
- Las features están disponibles antes de la decisión de admisión (no hay fuga del target).
- El problema se resuelve en modo offline/batch.
- El dataset es un snapshot estático (no hay proceso de actualización documentado).
- El RAW (`data/01_raw/`) permanece inmutable.
- Las filas duplicadas y los valores faltantes del RAW son parte del dato provisto y no se
  modifican en este Issue.

### 12. ¿Cuál es la fuente de los datos?
En el repositorio: `data/01_raw/Admission_Predict.csv`. La URL de origen, el autor y la
licencia **no están documentados** en el repositorio.

### 13. ¿Cómo se actualizan los datos?
**No documentado.** **Supuesto:** no existe un proceso de actualización; el archivo es un
snapshot estático.

### 14. ¿Cada cuánto tiempo se actualizan los datos?
**No documentado.** **Supuesto:** sin actualización periódica.


### Fuera de alcance en este Issue

No se realizan en el Issue #1: limpieza, tratamiento de valores faltantes, outliers,
feature engineering, escalado, encoding, separación train/test, baseline, entrenamiento de
modelos, despliegue ni automatización de pipelines. Este notebook solo lee y describe el RAW.


## 📚 Import libraries

In [1]:
import sys
from pathlib import Path

import pandas as pd

print(sys.executable)
print(sys.version)

/home/elkiruvi/Proyecto-Admisiones/.venv/bin/python3
3.12.13 (main, Aug  5 2026, 15:44:22) [Clang 22.1.3 ]


## 💾 Load data

Lectura **reproducible** e **inmutable** del RAW: se localiza la raíz del repositorio y se lee
`data/01_raw/Admission_Predict.csv` sin modificarlo.


In [2]:
def find_repo_root(start: Path) -> Path:
    """Localiza la raíz del repositorio subiendo hasta encontrar `pyproject.toml`."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del repositorio (pyproject.toml).")


ROOT = find_repo_root(Path.cwd())
RAW_PATH = ROOT / "data" / "01_raw" / "Admission_Predict.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset en {RAW_PATH}.")

# Solo lectura: el RAW no se modifica. Las celdas vacías se leen como NaN.
raw_df = pd.read_csv(RAW_PATH)

# 'LOR ' y 'Chance of Admit ' traen espacio final; se normaliza SOLO en memoria.
raw_df.columns = raw_df.columns.str.strip()

print(f"Dataset cargado desde: {RAW_PATH}")
raw_df.head()

Dataset cargado desde: /home/elkiruvi/Proyecto-Admisiones/data/01_raw/Admission_Predict.csv


,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
1,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
2,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
3,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
4,314.0,103.0,2.0,2.0,3.0,8.21,0.0,0.65


## 🔍 Exploración inicial mínima

Se identifica la variable objetivo y las features, y se documentan los valores faltantes.

- **Target:** `Chance of Admit` (probabilidad de admisión, 0–1).
- **Features:** `GRE Score`, `TOEFL Score`, `University Rating`, `SOP`, `LOR`, `CGPA`, `Research`.


In [3]:
print(f"Shape (filas, columnas): {raw_df.shape}")
print(f"Columnas: {list(raw_df.columns)}")
print("\nTipos de datos:")
print(raw_df.dtypes)
print("\nValores faltantes por columna:")
print(raw_df.isna().sum())
print("\nFilas duplicadas (completas):", int(raw_df.duplicated().sum()))
print("\nEstadísticas del target (Chance of Admit):")
print(raw_df["Chance of Admit"].describe())

Shape (filas, columnas): (623, 8)
Columnas: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit']

Tipos de datos:
GRE Score            float64
TOEFL Score          float64
University Rating    float64
SOP                  float64
LOR                  float64
CGPA                 float64
Research             float64
Chance of Admit      float64
dtype: object

Valores faltantes por columna:
GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0
dtype: int64

Filas duplicadas (completas): 152

Estadísticas del target (Chance of Admit):
count    623.000000
mean       0.733034
std        0.142240
min        0.340000
25%        0.640000
50%        0.730000
75%        0.840000
max        0.970000
Name: Chance of Admit, dtype: float64


## 📊 Análisis de resultados y conclusiones

- La variable objetivo `Chance of Admit` es continua (0–1), por lo que el problema es de
  **regresión supervisada** (batch/offline).
- El dataset RAW tiene 623 filas y 8 columnas; 7 de 8 columnas presentan valores faltantes,
  que aquí solo se **documentan** y se tratarán en etapas posteriores.
- El target tiene media ≈ 0.733 y desviación estándar ≈ 0.142; estos valores sirven de
  referencia para el baseline (ver pregunta 7).
- El RAW (`data/01_raw/`) permanece inmutable: este notebook solo lo lee.


## 💡 Propuestas e ideas (próximos pasos)

- Exploración y análisis exploratorio de datos (EDA).
- Tratamiento de valores faltantes y validación de tipos.
- Feature engineering justificado por el EDA.
- Baseline y modelamiento, con separación train/test y validación cruzada.


## 📖 Referencias

- `data/01_raw/Informacion.txt` — descripción del dataset y de sus variables.
- `AGENTS.md` — reglas del proyecto (datos RAW, notebooks, modelamiento, etc.).
- `notebooks/README.md` — convención de nombres de notebooks.
- `data/README.md` — convención de capas de datos (RAW inmutable).
- Dataset *Graduate Admissions* (atribución **no confirmada** por el repositorio; a verificar):
  <https://www.kaggle.com/datasets/mohansacharya/graduate-admissions>
